# Order Book Signals on Bond & Equity Calendar Spreads

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, '..')

import polars as pl
import numpy as np
from plotnine import *
from plotnine.themes import theme_bw
import pandas as pd

pl.Config.set_tbl_cols(30)
pl.Config.set_tbl_rows(20)

In [ ]:
from src.pipeline import (
    build_datasets, generate_signals, attach_leg_signals,
    TRAIN_YEARS, VAL_YEARS, TEST_YEARS, YEAR_LABEL,
)

df_cs, df_combined = build_datasets(env_path='../.env')   # or pass session=snowpark to reuse a connection
df_signals = generate_signals(df_cs)                       # replaces cells 10–11
# Augment each spread bin with the BUY/SELL leg outright OBI/STV (resolved via
# IS_CONVENTION_BUY_NEAR) -> adds buy_obi / buy_stv / sell_obi / sell_stv.
df_signals = attach_leg_signals(df_signals, df_combined)

In [ ]:
from src.ordered_logit import split_tick_constrained

df_signals_tu, df_signals_tc = split_tick_constrained(df_signals)
df_signals_tc.describe()

# Tick-constrained curve groups

Tick-constrained groups trade on a near-discrete price grid, so $\Delta P_t$ is cleaned to the
ordered set $\{-2, 0, +2\}$ (neighbour-context rules applied within each `[security, date]`
session over the full 2021-2025 sample) and modelled with a **class-weighted ordered logit**.

**Out-of-sample protocol.** Both the contemporaneous and the 1-step-ahead models are
- **trained** on 2021-2023 only, weighting the $\pm2$ minority tails by their train inverse-frequency;
- **tuned** on validation (2024) — the $-2$ and $+2$ decision thresholds are chosen
  **independently**, each maximising that tail's **F$_\beta$** (default $\beta = 0.5$, i.e.
  precision-weighted) rather than using naive arg-max-probability assignment;
- **evaluated** on the held-out test set (2025) with the validation-selected thresholds.

In [ ]:
from src.ordered_logit import clean_delta_p_tc

df_signals_tc_clean = clean_delta_p_tc(df_signals_tc)
counts = df_signals_tc_clean['delta_p'].value_counts(sort=True)
print(counts)

## OOS Setup & Helpers (Tick-Constrained)

The model, threshold tuning and reporting helpers live in **`src/ordered_logit.py`** and are imported
above, so this notebook and the walk-forward backtest cannot drift apart:
- `fit_ordered_logit` — fit on **train (2021-2023)** with **cluster-robust SE** (clustered on the
  `[security, date]` session, so the stacked panel does not induce spurious dependence at
  security boundaries), then predict class probabilities for train / val / test;
- `tune_thresholds` — pick the $-2$ and $+2$ decision boundaries **independently** on
  **validation (2024)**, each maximising that tail's **F$_\beta$** (one-vs-rest; `beta < 1`
  favours precision);
- `assign_classes` — apply the two tuned tail thresholds (the more probable tail wins if both
  fire);
- `pr_threshold_df` / `confusion_tile` — plotnine precision-recall and confusion-matrix plots;
- `report_all_splits` — classification reports (precision / recall / F1 / support) per split.

In [ ]:
from src.ordered_logit import (
    CATS, TC_FEATURES, DEFAULT_BETA,
    fit_ordered_logit, tune_thresholds, assign_classes,
    pr_threshold_df, report_all_splits, confusion_tile,
)
from sklearn.metrics import fbeta_score
from datetime import date

# F-beta for the per-tail threshold tuning. beta < 1 favours PRECISION over recall (a tail
# signal being right matters more than catching every tail). Vary it for ablation studies.
BETA = DEFAULT_BETA   # = 0.5

# Convert the pipeline's year-list split constants to (start, end) date pairs for fit_ordered_logit.
def _years_to_dates(years):
    return date(min(years), 1, 1), date(max(years), 12, 31)

TRAIN_SPLIT = _years_to_dates(TRAIN_YEARS)
VAL_SPLIT   = _years_to_dates(VAL_YEARS)
TEST_SPLIT  = _years_to_dates(TEST_YEARS)

## Contemporary Baseline: Ordered Logistic Regression (Tick-Constrained)

Class-weighted ordered logit with **cluster-robust** standard errors (clustered on the
`[security, date]` session, not a linear HAC across the stacked panel), **fit on train (2021-2023)**:

$$\Pr(\Delta P_t \le k) = \sigma(\alpha_k - \mathbf{x}_t^\top \boldsymbol{\beta}), \quad k \in \{-2, 0\}$$

Explanatory variables $\mathbf{x}_t = [OBI_t,\ NOI_t,\ STV_t]$; target $\Delta P_t \in \{-2, 0, 2\}$.
The tail-vs-majority decision threshold is tuned on validation (2024) and applied to test (2025).

In [ ]:
res_contemp, data_contemp, _ = fit_ordered_logit(
    df_signals_tc_clean, TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT,
    target_col='delta_p', label='Contemporaneous (tick-constrained)', verbose=True,
)

In [ ]:
# --- Tune the per-tail decision thresholds on VALIDATION (2024) -------------
# Tune the -2 and +2 boundaries INDEPENDENTLY to maximise each tail's F-beta; compare the
# achieved macro-F-beta to naive arg-max-prob assignment.
y_val_c, probs_val_c = data_contemp['val']

THR_DOWN_C, THR_UP_C, fb_dn_c, fb_up_c = tune_thresholds(y_val_c, probs_val_c, beta=BETA)
argmax_fb_c = fbeta_score(y_val_c, np.array(CATS)[probs_val_c.argmax(1)],
                          beta=BETA, labels=CATS, average='macro', zero_division=0)
print(f'Validation tail thresholds (β={BETA}):  -2 → {THR_DOWN_C:.3f} (Fβ={fb_dn_c:.4f})  |  '
      f'+2 → {THR_UP_C:.3f} (Fβ={fb_up_c:.4f})   vs  arg-max-prob macro-Fβ = {argmax_fb_c:.4f}')

# Precision/Recall vs threshold on VALIDATION, with each tail's selected boundary marked.
pr_long_c = pr_threshold_df(y_val_c, probs_val_c)
thr_lines_c = pd.DataFrame({'Class': ['Class -2', 'Class +2'], 'thr': [THR_DOWN_C, THR_UP_C]})
(
    ggplot(pr_long_c, aes(x='Threshold', y='Value', color='Metric'))
    + geom_line(size=1)
    + geom_vline(aes(xintercept='thr'), data=thr_lines_c, linetype='dashed', color='grey')
    + facet_wrap('~Class', scales='free_x')
    + theme_minimal()
    + labs(
        title=f'Validation Threshold Tuning — Contemporaneous (β={BETA}; '
              f'thr -2={THR_DOWN_C:.3f}, +2={THR_UP_C:.3f})',
        x='Probability Threshold Decision Boundary',
        y='Score Value',
        color='Metric',
    )
    + theme(figure_size=(10, 5))
)

In [ ]:
# --- Apply the validation-tuned thresholds: reports for all splits + TEST confusion matrix ---
# Train / val / test classification reports at the SAME validation-selected (THR_DOWN_C, THR_UP_C),
# then the held-out 2025 confusion matrix (raw counts + row %).
report_all_splits(data_contemp, THR_DOWN_C, THR_UP_C, 'Contemporaneous', split_labels=YEAR_LABEL)

y_test_c, probs_test_c = data_contemp['test']
confusion_tile(
    y_test_c, assign_classes(probs_test_c, THR_DOWN_C, THR_UP_C),
    title=f'Test (2025) Confusion — Ordered Logit Contemporaneous  '
          f'(thr -2={THR_DOWN_C:.3f}, +2={THR_UP_C:.3f})',
)

## Predictive Model: Ordered Logistic Regression (1-Step-Ahead)

Identical class-weighted ordered logit, but the target is $\Delta P_{t+1}$ — the next bin's
**cleaned** price change, shifted into the current row with `.shift(-1).over(['security', 'date'])`
in the cleaning step (before the split), so there is no look-ahead leakage across sessions or
years. Trained on 2021-2023, threshold tuned on validation (2024), evaluated on test (2025).

In [ ]:
res_fwd, data_fwd, _ = fit_ordered_logit(
    df_signals_tc_clean, TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT,
    target_col='delta_p_fwd', label='Predictive 1-step (tick-constrained)', verbose=True,
)

In [ ]:
# --- Tune the predictive per-tail decision thresholds on VALIDATION (2024) --
y_val_f, probs_val_f = data_fwd['val']

THR_DOWN_F, THR_UP_F, fb_dn_f, fb_up_f = tune_thresholds(y_val_f, probs_val_f, beta=BETA)
argmax_fb_f = fbeta_score(y_val_f, np.array(CATS)[probs_val_f.argmax(1)],
                          beta=BETA, labels=CATS, average='macro', zero_division=0)
print(f'Validation tail thresholds (β={BETA}):  -2 → {THR_DOWN_F:.3f} (Fβ={fb_dn_f:.4f})  |  '
      f'+2 → {THR_UP_F:.3f} (Fβ={fb_up_f:.4f})   vs  arg-max-prob macro-Fβ = {argmax_fb_f:.4f}')

# Precision/Recall vs threshold on VALIDATION, with each tail's selected boundary marked.
pr_long_f = pr_threshold_df(y_val_f, probs_val_f)
thr_lines_f = pd.DataFrame({'Class': ['Class -2', 'Class +2'], 'thr': [THR_DOWN_F, THR_UP_F]})
(
    ggplot(pr_long_f, aes(x='Threshold', y='Value', color='Metric'))
    + geom_line(size=1)
    + geom_vline(aes(xintercept='thr'), data=thr_lines_f, linetype='dashed', color='grey')
    + facet_wrap('~Class', scales='free_x')
    + theme_minimal()
    + labs(
        title=f'Validation Threshold Tuning — Predictive 1-Step (β={BETA}; '
              f'thr -2={THR_DOWN_F:.3f}, +2={THR_UP_F:.3f})',
        x='Probability Threshold Decision Boundary',
        y='Score Value',
        color='Metric',
    )
    + theme(figure_size=(10, 5))
)

In [ ]:
# --- Apply the validation-tuned thresholds: reports for all splits + TEST confusion matrix ---
report_all_splits(data_fwd, THR_DOWN_F, THR_UP_F, 'Predictive 1-step', split_labels=YEAR_LABEL)

y_test_f, probs_test_f = data_fwd['test']
confusion_tile(
    y_test_f, assign_classes(probs_test_f, THR_DOWN_F, THR_UP_F),
    title=f'Test (2025) Confusion — Ordered Logit Predictive 1-Step  '
          f'(thr -2={THR_DOWN_F:.3f}, +2={THR_UP_F:.3f})',
)

### Cross-`qcode` performance of the predictive model

The split-level reports above pool every tick-constrained curve group together. To check whether
the 1-step-ahead model is uniformly useful or only works on a subset, we break the **test (2025)**
tail performance out **by `qcode`**. The $+2$ and $-2$ classes are symmetric, so for each group
we report the **mean precision** and **mean recall** over the two tails (using the same fitted
model `res_fwd` and the same validation-tuned thresholds `THR_DOWN_F` / `THR_UP_F`). Groups are
ordered best → worst by $\tfrac12(\text{precision} + \text{recall})$; `tail_support` is the count
of actual $\pm2$ events, so thin bars on tiny support should be read with caution.

In [ ]:
# --- Per-qcode evaluation of the predictive model on TEST (2025) ----------
# Tail (+/-2) precision & recall, averaged over the two tail classes (they are symmetric),
# computed SEPARATELY for each qcode so we can see where the 1-step-ahead model
# generalises and where it degrades. Uses the SAME fitted model (res_fwd) and the SAME
# validation-tuned (THR_DOWN_F, THR_UP_F); we just re-materialise the test slice carrying
# qcode so every row can be grouped.
from sklearn.metrics import precision_recall_fscore_support

_pdf_bbg = (
    df_signals_tc_clean
    .filter(pl.col('date').dt.year().is_in(TEST_YEARS))
    .select(['delta_p_fwd', *TC_FEATURES, 'qcode'])
    .drop_nulls()
    .to_pandas()
)
_pdf_bbg['y']    = _pdf_bbg['delta_p_fwd'].astype(int)
_pdf_bbg['pred'] = assign_classes(
    np.asarray(res_fwd.predict(exog=_pdf_bbg[TC_FEATURES])), THR_DOWN_F, THR_UP_F
)

rows = []
for code, g in _pdf_bbg.groupby('qcode'):
    prec, rec, _, sup = precision_recall_fscore_support(
        g['y'], g['pred'], labels=[-2, 2], zero_division=0,
    )
    rows.append({
        'qcode': code,
        'Precision': float(prec.mean()),   # mean over the +2 and -2 classes
        'Recall': float(rec.mean()),
        'tail_support': int(sup.sum()),    # # of actual +/-2 observations
        'n': int(len(g)),
    })

bbg_metrics = (
    pd.DataFrame(rows)
    .assign(MeanPR=lambda d: (d['Precision'] + d['Recall']) / 2)
    .sort_values('MeanPR', ascending=False)
    .reset_index(drop=True)
)
print(bbg_metrics.to_string(index=False))

# Dodged precision/recall bars, qcode ordered best -> worst by mean(Precision, Recall).
_order   = bbg_metrics['qcode'].tolist()
_grand   = bbg_metrics[['Precision', 'Recall']].to_numpy().mean()
bbg_long = bbg_metrics.melt(
    id_vars=['qcode'], value_vars=['Precision', 'Recall'],
    var_name='Metric', value_name='Value',
)
bbg_long['qcode'] = pd.Categorical(bbg_long['qcode'], categories=_order, ordered=True)
(
    ggplot(bbg_long, aes(x='qcode', y='Value', fill='Metric'))
    + geom_col(position=position_dodge(width=0.8), width=0.7)
    + geom_hline(yintercept=_grand, linetype='dashed', color='grey')
    + scale_y_continuous(limits=[0, 1])
    + labs(
        title=f'Predictive 1-Step — Tail (±2) Precision/Recall by qcode  '
              f'(test 2025, β={BETA})',
        subtitle='Mean over the +2 and -2 classes; qcode ordered best → worst by '
                 'mean(Precision, Recall). Dashed line = grand mean.',
        x='qcode', y='Score', fill='Metric',
    )
    + theme_minimal()
    + theme(figure_size=(11, 5), axis_text_x=element_text(rotation=45, hjust=1))
)

## Predictive Model + Outright Legs: Ordered Logit (1-Step-Ahead, 6 Features)

The same class-weighted ordered logit and out-of-sample protocol as the predictive model above,
but the feature set is augmented with the **outright legs' microstructure**. Alongside the
spread's own $OBI_t$ / $STV_t$ we add the normalised $OBI$ / $STV$ of the **buy** and **sell**
legs — resolved per `qcode` from `IS_CONVENTION_BUY_NEAR` (buy = near or far leg accordingly) and
joined on the matching `[date, bin_start_time]` — giving six features:

$$\mathbf{x}_t = [\,OBI_t,\ STV_t,\ OBI^{\text{buy}}_t,\ STV^{\text{buy}}_t,\ OBI^{\text{sell}}_t,\ STV^{\text{sell}}_t\,].$$

**Economic prior:** a quote/flow improvement on the **buy** leg should push the calendar-spread
price **up** ($+2$), and an improvement on the **sell** leg should push it **down** ($-2$).

Trained on 2021-2023, thresholds tuned on validation (2024), evaluated on test (2025) — reported
**pooled** and **per-`qcode`**. Bins where a leg signal is missing (leg outside its roll window)
are dropped by the fitter's `drop_nulls`, so this model is fit and scored on common leg-support.

In [ ]:
LEG_FEATURES = ['obi', 'stv', 'buy_obi', 'buy_stv', 'sell_obi', 'sell_stv']
res_fwd_leg, data_fwd_leg, _ = fit_ordered_logit(
    df_signals_tc_clean, TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT,
    target_col='delta_p_fwd', feature_cols=LEG_FEATURES,
    label='Predictive 1-step + legs', verbose=True,
)

In [ ]:
# --- Tune the leg-augmented per-tail decision thresholds on VALIDATION (2024) ---
y_val_fl, probs_val_fl = data_fwd_leg['val']

THR_DOWN_FL, THR_UP_FL, fb_dn_fl, fb_up_fl = tune_thresholds(y_val_fl, probs_val_fl, beta=BETA)
argmax_fb_fl = fbeta_score(y_val_fl, np.array(CATS)[probs_val_fl.argmax(1)],
                           beta=BETA, labels=CATS, average='macro', zero_division=0)
print(f'Validation tail thresholds (β={BETA}):  -2 → {THR_DOWN_FL:.3f} (Fβ={fb_dn_fl:.4f})  |  '
      f'+2 → {THR_UP_FL:.3f} (Fβ={fb_up_fl:.4f})   vs  arg-max-prob macro-Fβ = {argmax_fb_fl:.4f}')

# Precision/Recall vs threshold on VALIDATION, with each tail's selected boundary marked.
pr_long_fl = pr_threshold_df(y_val_fl, probs_val_fl)
thr_lines_fl = pd.DataFrame({'Class': ['Class -2', 'Class +2'], 'thr': [THR_DOWN_FL, THR_UP_FL]})
(
    ggplot(pr_long_fl, aes(x='Threshold', y='Value', color='Metric'))
    + geom_line(size=1)
    + geom_vline(aes(xintercept='thr'), data=thr_lines_fl, linetype='dashed', color='grey')
    + facet_wrap('~Class', scales='free_x')
    + theme_minimal()
    + labs(
        title=f'Validation Threshold Tuning — Predictive 1-Step + Legs (β={BETA}; '
              f'thr -2={THR_DOWN_FL:.3f}, +2={THR_UP_FL:.3f})',
        x='Probability Threshold Decision Boundary',
        y='Score Value',
        color='Metric',
    )
    + theme(figure_size=(10, 5))
)

In [ ]:
# --- Pooled evaluation: reports for all splits + TEST confusion matrix ------
report_all_splits(data_fwd_leg, THR_DOWN_FL, THR_UP_FL, 'Predictive 1-step + legs',
                  split_labels=YEAR_LABEL)

y_test_fl, probs_test_fl = data_fwd_leg['test']
confusion_tile(
    y_test_fl, assign_classes(probs_test_fl, THR_DOWN_FL, THR_UP_FL),
    title=f'Test (2025) Confusion — Ordered Logit Predictive 1-Step + Legs  '
          f'(thr -2={THR_DOWN_FL:.3f}, +2={THR_UP_FL:.3f})',
)

### Cross-`qcode` comparison: does adding the legs help?

To isolate the **incremental** value of the outright-leg features, both models are scored on the
**same test (2025) rows** — those where every leg feature is present, so the 6-feature model is
defined (`common leg-support`). For each `qcode` we report the tail ($\pm2$) **mean precision /
recall**, then compare the base 3-feature predictive model against the leg-augmented 6-feature
one. A positive `Delta` means the legs improve that curve group; this surfaces where the
buy/sell-leg microstructure carries genuine predictive signal versus where it adds nothing.

In [ ]:
# --- Per-qcode: base (3f) vs leg-augmented (6f) on COMMON leg-support (test 2025) ---
# Both models are scored on the SAME rows (every leg feature present) so the comparison
# reflects only the added features, not a change in the evaluated sample. Each model uses its
# OWN validation-tuned per-tail thresholds. Tail (+/-2) precision/recall are averaged over the
# two symmetric classes within each qcode.
ALL_FEATS = sorted(set(TC_FEATURES) | set(LEG_FEATURES))   # union: obi, noi, stv, buy_*, sell_*

_cmp = (
    df_signals_tc_clean
    .filter(pl.col('date').dt.year().is_in(TEST_YEARS))
    .select(['delta_p_fwd', *ALL_FEATS, 'qcode'])
    .drop_nulls()
    .to_pandas()
)
_cmp['y'] = _cmp['delta_p_fwd'].astype(int)
_cmp['pred_base'] = assign_classes(
    np.asarray(res_fwd.predict(exog=_cmp[TC_FEATURES])), THR_DOWN_F, THR_UP_F)
_cmp['pred_leg']  = assign_classes(
    np.asarray(res_fwd_leg.predict(exog=_cmp[LEG_FEATURES])), THR_DOWN_FL, THR_UP_FL)


def _tail_pr(g, pred_col):
    """Mean precision/recall over the two tail classes (+/-2) for one qcode group."""
    prec, rec, _, sup = precision_recall_fscore_support(
        g['y'], g[pred_col], labels=[-2, 2], zero_division=0)
    return {'Precision': float(prec.mean()), 'Recall': float(rec.mean()),
            'MeanPR': float((prec.mean() + rec.mean()) / 2), 'tail_support': int(sup.sum())}

rows = []
for code, g in _cmp.groupby('qcode'):
    for model, pc in (('Base (3f)', 'pred_base'), ('+Legs (6f)', 'pred_leg')):
        rows.append({'qcode': code, 'Model': model, **_tail_pr(g, pc), 'n': int(len(g))})
cmp_metrics = pd.DataFrame(rows)

# Wide MeanPR table with the base -> legs delta, ordered by the leg model.
wide = (
    cmp_metrics.pivot_table(index='qcode', columns='Model', values='Precision')
    .assign(Delta=lambda d: d['+Legs (6f)'] - d['Base (3f)'])
    .sort_values('+Legs (6f)', ascending=False)
)
print(wide.to_string(float_format=lambda x: f'{x:.3f}'))

# Dodged bars: base vs legs per qcode (ordered best -> worst by the leg model's MeanPR).
_order = wide.index.tolist()
cmp_metrics['qcode'] = pd.Categorical(cmp_metrics['qcode'], categories=_order, ordered=True)
cmp_metrics['Model'] = pd.Categorical(cmp_metrics['Model'], categories=['Base (3f)', '+Legs (6f)'])
(
    ggplot(cmp_metrics, aes(x='qcode', y='Precision', fill='Model'))
    + geom_col(position=position_dodge(width=0.8), width=0.7)
    + scale_y_continuous(limits=[0, 1])
    + labs(
        title=f'Per-qcode Tail (±2) Precision — Base vs +Legs  '
              f'(test 2025, β={BETA}, common leg-support, n={len(_cmp):,})',
        subtitle='Mean over the +2 and -2 classes; both models scored on identical rows. '
                 'qcode ordered best → worst by the +Legs score.',
        x='qcode', y='Mean(Precision, Recall)', fill='Model',
    )
    + theme_minimal()
    + theme(figure_size=(11, 5), axis_text_x=element_text(rotation=45, hjust=1))
)